# PyGeoModel Refactor Test Record

Generated for the `codex/pygeomodel-core-api-refactor` branch.

This notebook records a step-by-step validation of the refactored PyGeoModel package. It separates offline tests, which should always pass, from optional online integration tests that require credentials and OpenGMS network access.

Generated at: 2026-05-10T14:43:35


## 1. Environment and import check

Goal: verify that the refactored package imports from the new `pygeomodel/` package structure and exposes the public API.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

print('Python:', sys.version)
print('CWD:', Path.cwd())
try:
    print('Git branch:', subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())
    print('Git HEAD:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
except Exception as exc:
    print('Git info unavailable:', exc)

from pygeomodel import (
    GeoModeler,
    ModelService,
    TaskResult,
    RecommendationResult,
    QAResult,
    OpenGMSClient,
)
import pygeomodel

print('pygeomodel module:', pygeomodel.__file__)
print('public imports: OK')


## 2. Local model catalog loading

Goal: verify that the bundled OpenGMS model catalog can be loaded after moving it into `pygeomodel/data/`.

In [ ]:
modeler = GeoModeler()
print('Model count:', len(modeler.model_names))
assert len(modeler.model_names) == 4786, 'Expected 4786 bundled model records'
print('Catalog loading: OK')

## 3. Model search API

Goal: verify `search_models()` returns structured model summaries for a realistic query.

In [ ]:
results = modeler.search_models('photovoltaic', limit=5)
for idx, item in enumerate(results, start=1):
    print(f'{idx}. {item.name}')
    print('   description:', (item.description or '')[:180].replace('\n', ' '))

assert results, 'search_models should return at least one photovoltaic-related model'
assert any('Photovoltaic' in item.name for item in results), 'Expected photovoltaic model in results'
print('Model search: OK')

## 4. Model metadata inspection

Goal: verify `get_model()` returns a `ModelService` with parsed inputs, outputs, state names, and data types.

In [ ]:
pv_name = 'Roof Photovoltaic Carbon Emission Reduction Potential Assessment Model'
pv_model = modeler.get_model(pv_name)
print('Model:', pv_model.name)
print('Description:', pv_model.description[:300].replace('\n', ' '))
print('Inputs:', len(pv_model.inputs))
print('Outputs:', len(pv_model.outputs))

for item in pv_model.inputs:
    print('INPUT', item.to_dict())
for item in pv_model.outputs:
    print('OUTPUT', item.to_dict())

assert isinstance(pv_model, ModelService)
assert any(item.name == 'system_efficiency' and item.data_type == 'REAL' for item in pv_model.inputs)
assert any(item.name == 'roof_vector_path' and item.is_file for item in pv_model.inputs)
print('Metadata inspection: OK')


## 5. Parameter normalization without network access

Goal: verify flat Python parameters are converted into the OpenGMS state/event structure, with numeric parameters typed through `dataType` and relative/absolute files treated as files.

In [ ]:
from tempfile import TemporaryDirectory
from pathlib import Path

with TemporaryDirectory() as tmpdir:
    roof_file = Path(tmpdir) / 'rooftops.zip'
    roof_file.write_bytes(b'fake rooftop data for parameter-normalization test')
    normalized = pv_model.normalize_params({
        'system_efficiency': '0.8',
        'start_time': '201801',
        'end_time': '201812',
        'roof_vector_path': str(roof_file),
    })

print(normalized)
assert normalized['SpatialAnalysis']['system_efficiency'] == 0.8
assert normalized['SpatialAnalysis']['start_time'] == 201801.0
assert normalized['SpatialAnalysis']['end_time'] == 201812.0
assert normalized['SolarCalculation']['roof_vector_path'].endswith('rooftops.zip')
print('Parameter normalization: OK')


## 6. Invocation API with a fake OpenGMS client

Goal: verify `invoke()` returns a `TaskResult` and records the normalized state/event input structure without calling OpenGMS.

In [ ]:
class FakeClient:
    def __init__(self):
        self.calls = []
        self.manager_url = 'mock://manager'

    def create_task(self, model_name, params, wait=True):
        self.calls.append((model_name, params, wait))
        return {
            'task_id': 'task-123',
            'status': 'completed',
            'outputs': [{
                'statename': 'SolarCalculation',
                'event': 'roofSloar',
                'url': 'http://example.com/result.zip',
                'suffix': 'zip',
                'tag': 'roofSloar',
            }],
            'execution_time': 0.01,
        }

with TemporaryDirectory() as tmpdir:
    roof_file = Path(tmpdir) / 'rooftops.zip'
    roof_file.write_bytes(b'fake rooftop data')
    fake_modeler = GeoModeler(client=FakeClient())
    result = fake_modeler.invoke(
        pv_name,
        params={
            'system_efficiency': 0.8,
            'start_time': 201801,
            'end_time': 201812,
            'roof_vector_path': str(roof_file),
        },
    )

print(result.to_dict())
assert isinstance(result, TaskResult)
assert result.task_id == 'task-123'
assert result.status == 'completed'
assert result.outputs[0]['event'] == 'roofSloar'
assert fake_modeler.last_result is result
print('Invocation with fake client: OK')


## 7. Structured record serialization

Goal: verify `TaskResult`, `RecommendationResult`, and `QAResult` can be exported as JSON records.

In [ ]:
import json
from tempfile import TemporaryDirectory

with TemporaryDirectory() as tmpdir:
    tmpdir = Path(tmpdir)

    task_path = result.to_json(tmpdir / 'execution_record.json')
    rec = RecommendationResult(
        primary_model={'name': pv_name},
        candidates=[{'name': 'candidate'}],
        recommended_data={'local_data': []},
        context={'modeling_history': 'test context'},
    )
    rec_path = rec.to_json(tmpdir / 'recommendation_record.json')

    qa = QAResult(
        question='What input data are required?',
        answer='A rooftop vector dataset and scalar time/efficiency parameters are required.',
        model_name=pv_name,
        sources=[{'type': 'metadata'}],
    )
    qa_path = qa.to_json(tmpdir / 'qa_record.json')

    for path in [task_path, rec_path, qa_path]:
        assert Path(path).exists()
        payload = json.loads(Path(path).read_text(encoding='utf-8'))
        print(path, payload.keys())

print('Record serialization: OK')


## 8. Recommendation fallback record

Goal: verify `suggest_model()` returns a `RecommendationResult` even without Dify credentials, preserving context and a local metadata-based fallback.

In [ ]:
rec = modeler.suggest_model(
    context='Assess rooftop photovoltaic potential in Nanjing using rooftop vector data.',
    data_context='A zipped rooftop polygon dataset is available.',
    include_trace=True,
    return_result=True,
)

print(rec.to_dict())
assert isinstance(rec, RecommendationResult)
assert rec.primary_model
assert 'context' in rec.to_dict()
print('Recommendation fallback: OK')


## 9. Q&A metadata fallback

Goal: verify `ask_model()` returns a `QAResult` without requiring OpenAI credentials.

In [ ]:
qa = modeler.ask_model(pv_name, 'What input data are required for this model?')
print(qa.to_dict())
assert isinstance(qa, QAResult)
assert qa.answer
assert qa.model_name == pv_name
print('Q&A fallback: OK')


## 10. Notebook widget smoke test

Goal: verify the notebook interface can create widgets. This test does not click Run or call OpenGMS.

In [ ]:
widget = modeler.invoke_model(pv_name)
print(type(widget))
print('children:', len(widget.children))
assert hasattr(widget, 'children')
print('Widget smoke test: OK')


## 11. Optional online OpenGMS checks

Goal: check OpenGMS token availability and service status. This section is skipped when `OGMS_TOKEN` is not set.

In [ ]:
import os

if not os.environ.get('OGMS_TOKEN'):
    print('SKIPPED: OGMS_TOKEN is not configured. Set OGMS_TOKEN to run online invocation checks.')
else:
    client = OpenGMSClient()
    print('Token valid:', client.validate_token())
    print('PV service ready:', client.check_model_service(pv_name))


## 12. Distribution and unit-test commands

Goal: record the command-line checks that should also be run outside the notebook.

In [ ]:
import subprocess
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / 'setup.py').exists() else cwd.parent
assert (repo_root / 'setup.py').exists(), f'Cannot locate repository root from {cwd}'

commands = [
    ['python', '-m', 'unittest', 'discover', '-s', 'tests'],
    [
        'python',
        '-c',
        "from pygeomodel import GeoModeler; m=GeoModeler(); print(len(m.model_names)); print(m.search_models('photovoltaic', limit=1)[0].name)",
    ],
]

for cmd in commands:
    print('$', ' '.join(cmd))
    completed = subprocess.run(
        cmd, cwd=repo_root, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print(completed.stdout)
    assert completed.returncode == 0

print('Command-line checks: OK')


## Summary

If all offline sections pass, the refactored core API is working locally. Any skipped online checks should be revisited after configuring `OGMS_TOKEN` and, if needed, `DIFY_API_KEY` / `OPENAI_API_KEY`.

In [1]:
%pip install -e /Users/mpl/Downloads/coding/project/work/PyGeoModel

from pygeomodel import GeoModeler
import pygeomodel

modeler = GeoModeler()
modeler.show_models()

Obtaining file:///Users/mpl/Downloads/coding/project/work/PyGeoModel
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for PyGeoModel (pyproject.toml) ... done
  Created wheel for PyGeoModel: filename=pygeomodel-1.0.13-0.editable-py3-none-any.whl size=5208 sha256=ac9aae4b435e4a0d3239bfe12a6daa534fa414787ad74ff71cd1956592bd024b
  Stored in directory: /private/var/folders/51/pccwx9rs17j2q432g1011yqh0000gn/T/pip-ephem-wheel-cache-zgtdkizl/wheels/3a/2b/de/36cff4bfe1bd2cdaafa787be59d13b0f023265c6030563409b
Successfully built PyGeoModel
  Attempting uninstall: PyGeoModel
    Found existing installation: PyGeoModel 1.0.13
    Uninstalling PyGeoModel-1.0.13:
      Successfully uninstalled PyGeoModel-1.0.13

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
N

In [ ]:
print(pygeomodel.__file__)
modeler = GeoModeler()
len(modeler.model_names)

In [ ]:
rec = modeler.suggest_model()
rec

In [ ]:
len(modeler.model_names)

In [ ]:
modeler.search_models("绝对湿度", limit=5)

In [ ]:
modeler.search_models("photovoltaic", limit=5)

In [ ]:
model = modeler.get_model("绝对湿度模型")
model.name
model.description
model.inputs
model.outputs
model.to_dict()

In [ ]:
modeler.invoke_model("绝对湿度模型")

In [ ]:
result.status

In [ ]:
result = modeler.invoke(
    "绝对湿度模型",
    {"ea": "3", "Mm": "2", "R": "1", "T": "2"}
)

In [ ]:
result.status

In [ ]:
result.task_id

In [ ]:
result.outputs


In [ ]:
result.download("./data/results")

In [ ]:
answer = modeler.ask_model(
    "绝对湿度模型",
    "这个模型的输入参数分别是什么意思？"
)
answer

In [ ]:
answer = modeler.ask_model(
    "绝对湿度模型",
    "普适气体常数是啥"
)
answer